In [28]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

# 1629 Características del hogar


In [29]:
# RECH0 es una base a nivel hogar
# RECH1 Es la base a nivel persona del hogar.
# RECH4 Está a nivel persona
# RECHM mortalidad
# Para nuestros fines nos conviene usar RECH0 y RECH1

In [30]:
base = Path("../data/demografica_salud_ENDES")

ruta_rech0 = base / "1629" / "RECH0_2024.csv"
ruta_rech1 = base / "1629" / "RECH1_2024.csv"

rech0 = pd.read_csv(ruta_rech0, dtype=str)
rech1 = pd.read_csv(ruta_rech1, dtype=str)

In [31]:
rech0

,ID1,HHID,HV000,HV001,HV002,HV002A,HV003,HV004,HV007,HV008,...,HV043,HV044,UBIGEO,HV022,HV005,CODCCPP,NOMCCPP,LATITUDY,LONGITUDX,NCONGLOME1
0,2024,325502001,PE6,3255,20,1,0,3255,2024,1493,...,0,1,010101,2,0,0001,CHACHAPOYAS,-6.2258317,-77.8613021,07076
1,2024,325503101,PE6,3255,31,1,2,3255,2024,1493,...,0,1,010101,2,74497,0001,CHACHAPOYAS,-6.2258317,-77.8613021,07076
2,2024,325503901,PE6,3255,39,1,1,3255,2024,1493,...,0,1,010101,2,607697,0001,CHACHAPOYAS,-6.2258317,-77.8613021,07076
3,2024,325504001,PE6,3255,40,1,1,3255,2024,1493,...,0,1,010101,2,607697,0001,CHACHAPOYAS,-6.2258317,-77.8613021,07076
4,2024,325504701,PE6,3255,47,1,2,3255,2024,1493,...,0,1,010101,2,74497,0001,CHACHAPOYAS,-6.2258317,-77.8613021,07076
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37385,2024,705704301,PE6,7057,43,1,1,7057,2024,1500,...,0,1,110101,100,251377,0001,ICA,-14.03525917,-75.737065,3426301
37386,2024,705705001,PE6,7057,50,1,2,7057,2024,1500,...,0,1,110101,100,251377,0001,ICA,-14.03525917,-75.737065,3426301
37387,2024,705706501,PE6,7057,65,1,2,7057,2024,1500,...,0,1,110101,100,251377,0001,ICA,-14.03525917,-75.737065,3426301
37388,2024,705707301,PE6,7057,73,1,0,7057,2024,1500,...,0,1,110101,100,0,0001,ICA,-14.03525917,-75.737065,3426301


In [ ]:
rech0['UBIGEO'] = rech0['UBIGEO'].astype(str).str.zfill(6)
rech0["ID1"] = rech0["ID1"].astype(int)


In [13]:
rech0["UBIGEO"].unique()

array(['010101', '010202', '010201', '010701', '010706', '010306',
       '010401', '010205', '010704', '010301', '010504', '010517',
       '010104', '010109', '010605', '021801', '021809', '020105',
       '020101', '020801', '020701', '021004', '021804', '021511',
       '021909', '021304', '020606', '021003', '030101', '030201',
       '030104', '030601', '030605', '030219', '030207', '030209',
       '030107', '030415', '030404', '030704', '030504', '040117',
       '040101', '040126', '040110', '040112', '040129', '040107',
       '040123', '040109', '040104', '040208', '040520', '040704',
       '040410', '040106', '050101', '050104', '050110', '050401',
       '050507', '050201', '050701', '050402', '050203', '051101',
       '051012', '050406', '050604', '050617', '050619', '051102',
       '060101', '060801', '060601', '060904', '060407', '060307',
       '060304', '060703', '060701', '061112', '061105', '061007',
       '060203', '070106', '070107', '070103', '070102', '0701

In [7]:
# Nos quedaremos solo con las vairables
cols_keep = [
    "ID1",     # año
    "HHID",    # id hogar
    "UBIGEO",  # distrito
    "HV001",   # conglomerado
    "HV022",   # estrato
    "HV024",   # región/departamento
    "HV025",   # urbano/rural
    "HV005",   # peso hogar
    "HV009",   # tamaño hogar
    "HV014"    # niños <5
]

rech0 = rech0[cols_keep].copy()

In [19]:
rech0

,ID1,HHID,UBIGEO,HV001,HV022,HV024,HV025,HV005,HV009,HV014
0,2024,325502001,010101,3255,2,1,1,0,0,0
1,2024,325503101,010101,3255,2,1,1,74497,4,1
2,2024,325503901,010101,3255,2,1,1,607697,2,0
3,2024,325504001,010101,3255,2,1,1,607697,1,0
4,2024,325504701,010101,3255,2,1,1,74497,4,1
...,...,...,...,...,...,...,...,...,...,...
37385,2024,705704301,110101,7057,100,11,1,251377,4,1
37386,2024,705705001,110101,7057,100,11,1,251377,10,2
37387,2024,705706501,110101,7057,100,11,1,251377,4,1
37388,2024,705707301,110101,7057,100,11,1,0,0,0


In [20]:
rech0.dtypes

cat = ["ID1", "HHID", "UBIGEO", "HV001", "HV022", "HV024", "HV025"]
num = ["HV005", "HV009", "HV014"]
rech0[cat] = rech0[cat].astype("category")
rech0[num] = rech0[num].astype(float)

In [21]:
# uniremos por unigeo
rech0_dist = (
    rech0.groupby("UBIGEO").apply(lambda x: pd.Series({
        "ID1": 2024,
        "n_hogares_muestra": x["HHID"].count(),
        "hogares_ponderados": x["HV005"].sum(),
        "promedio_personas_hogar": (x["HV009"] * x["HV005"]).sum() / x["HV005"].sum(),
        "proporcion_hogares_rurales": ((x["HV025"] == 2) * x["HV005"]).sum() / x["HV005"].sum(),
        "proporcion_hogares_con_ninos_menores_5": ((x["HV014"] > 0) * x["HV005"]).sum() / x["HV005"].sum()
    }))
    .reset_index()
)

C:\Users\JHOSSEP\AppData\Local\Temp\ipykernel_3524\1914414122.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  rech0.groupby("UBIGEO").apply(lambda x: pd.Series({


In [22]:
rech0_dist

,UBIGEO,ID1,n_hogares_muestra,hogares_ponderados,promedio_personas_hogar,proporcion_hogares_rurales,proporcion_hogares_con_ninos_menores_5
0,010101,2024.0,141.0,32949454.0,3.467306,0.0,0.270722
1,010103,2024.0,15.0,5375984.0,2.987279,0.0,0.368916
2,010104,2024.0,15.0,3985760.0,3.591590,0.0,0.248848
3,010109,2024.0,15.0,3421492.0,3.442839,0.0,0.283292
4,010112,2024.0,15.0,3296214.0,2.539169,0.0,0.342741
...,...,...,...,...,...,...,...
920,250301,2024.0,111.0,46015830.0,3.701278,0.0,0.390925
921,250302,2024.0,35.0,13841942.0,3.529395,0.0,0.459384
922,250303,2024.0,15.0,6288856.0,4.661157,0.0,0.548209
923,250305,2024.0,25.0,11511454.0,4.181307,0.0,0.435935


In [24]:
rech0_dist.isnull().sum()

UBIGEO                                    0
ID1                                       0
n_hogares_muestra                         0
hogares_ponderados                        0
promedio_personas_hogar                   0
proporcion_hogares_rurales                0
proporcion_hogares_con_ninos_menores_5    0
dtype: int64

In [27]:
rech0_dist.describe()

,ID1,n_hogares_muestra,hogares_ponderados,promedio_personas_hogar,proporcion_hogares_rurales,proporcion_hogares_con_ninos_menores_5
count,925.0,925.000000,9.250000e+02,925.000000,925.0,925.000000
mean,2024.0,40.421622,3.677622e+07,3.302969,0.0,0.290545
std,0.0,64.137346,8.621039e+07,0.625868,0.0,0.132147
min,2024.0,10.000000,1.304622e+06,1.331039,0.0,0.000000
25%,2024.0,15.000000,8.219112e+06,2.890119,0.0,0.204366
50%,2024.0,20.000000,1.437744e+07,3.293723,0.0,0.278066
75%,2024.0,40.000000,2.994194e+07,3.659864,0.0,0.361622
max,2024.0,777.000000,1.313308e+09,6.105340,0.0,1.000000


In [ ]:
rech1
# esta es una base a nivel del hogar, entonces debemos resumirla a nivel hogar y luego unirla con rech0 cpor distrito

,ID1,HHID,HVIDX,HV101,HV102,HV103,HV104,HV105,HV106,HV107,...,HV125,HV126,HV127,HV128,HV129,QH21A,QH25A,QH25B,QH25CM,QH25CA
0,2024,325503101,1,1,1,1,1,35,1,6,...,0,0,,0,,,PERUANA,,,
1,2024,325503101,2,2,1,1,2,38,2,5,...,0,0,,0,,,PERUANA,,,
2,2024,325503101,3,3,1,1,2,3,0,,...,0,0,,0,,1,PERUANA,,,
3,2024,325503101,4,11,1,1,2,15,2,3,...,1,2,3,9,2,,PERUANA,,,
4,2024,325503901,1,1,1,1,2,53,2,5,...,0,0,,0,,,PERUANA,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
135040,2024,705706501,4,11,1,1,1,12,1,6,...,1,1,6,6,4,,PERUANA,,,
135041,2024,705708801,1,1,1,1,2,39,3,3,...,0,0,,0,,,PERUANA,,,
135042,2024,705708801,2,3,1,1,1,16,2,4,...,1,2,4,10,2,,PERUANA,,,
135043,2024,705708801,3,3,1,1,2,13,1,6,...,1,1,6,6,4,,PERUANA,,,


In [33]:
rech1.shape

(135045, 36)

In [34]:
vars_rech1 = [
    "HHID", "HVIDX",
    "HV101", "HV102", "HV103",
    "HV104", "HV105",
    "HV106", "HV107", "HV108", "HV109",
    "HV121", "HV122", "HV123", "HV124",
    "HV125", "HV126", "HV127", "HV128", "HV129",
    "QH21A", "QH25B"
]

rech1 = rech1[[col for col in vars_rech1 if col in rech1.columns]].copy()

In [35]:
vars_numericas = [
    "HVIDX",
    "HV101", "HV102", "HV103",
    "HV104", "HV105",
    "HV106", "HV107", "HV108", "HV109",
    "HV121", "HV122", "HV123", "HV124",
    "HV125", "HV126", "HV127", "HV128", "HV129",
    "QH21A", "QH25B"
]

vars_categoricas = [col for col in vars_rech1 if col not in vars_numericas]

rech1[vars_numericas] = rech1[vars_numericas].apply(pd.to_numeric, errors='coerce')
rech1[vars_categoricas] = rech1[vars_categoricas].astype('category')

In [36]:
rech1

,HHID,HVIDX,HV101,HV102,HV103,HV104,HV105,HV106,HV107,HV108,...,HV122,HV123,HV124,HV125,HV126,HV127,HV128,HV129,QH21A,QH25B
0,325503101,1,1,1,1,1,35,1,6.0,6,...,0,NaN,0,0,0,NaN,0,NaN,NaN,NaN
1,325503101,2,2,1,1,2,38,2,5.0,11,...,0,NaN,0,0,0,NaN,0,NaN,NaN,NaN
2,325503101,3,3,1,1,2,3,0,NaN,0,...,0,NaN,0,0,0,NaN,0,NaN,1.0,NaN
3,325503101,4,11,1,1,2,15,2,3.0,9,...,2,4.0,10,1,2,3.0,9,2.0,NaN,NaN
4,325503901,1,1,1,1,2,53,2,5.0,11,...,0,NaN,0,0,0,NaN,0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
135040,705706501,4,11,1,1,1,12,1,6.0,6,...,0,NaN,0,1,1,6.0,6,4.0,NaN,NaN
135041,705708801,1,1,1,1,2,39,3,3.0,14,...,0,NaN,0,0,0,NaN,0,NaN,NaN,NaN
135042,705708801,2,3,1,1,1,16,2,4.0,10,...,2,5.0,11,1,2,4.0,10,2.0,NaN,NaN
135043,705708801,3,3,1,1,2,13,1,6.0,6,...,0,NaN,0,1,1,6.0,6,4.0,NaN,NaN


# 1630

In [16]:
ruta_rech23 = base / "1630" / "RECH23_2024.csv"

rech23 = pd.read_csv(ruta_rech23, dtype=str)

In [17]:
rech23

,ID1,HHID,HV201,HV202,HV204,HV205,HV206,HV207,HV208,HV209,...,SH78,SH79,SH224,SH225U,SH225,SH227,QH227A,QH227B,HV270,HV271
0,2024,325502001,,,,,,,,,...,,,,,,,,,,
1,2024,325503101,11,,996,22,1,0,0,0,...,0,,9,,,2,1,1,2,-.224175006493913
2,2024,325503901,11,,996,11,1,0,0,0,...,0,,9,,,3,1,1,3,.366367930920724
3,2024,325504001,11,,996,22,1,1,0,0,...,0,,9,,,3,1,1,1,-.76953388723092
4,2024,325504701,71,11,996,11,1,1,1,1,...,0,,9,,,5,,,2,.0566849954924239
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37385,2024,705704301,71,61,1,21,1,1,1,1,...,0,,9,,,5,,,2,.212642980267205
37386,2024,705705001,96,96,1,21,1,1,1,0,...,0,,9,,,3,1,1,2,-.519781023549353
37387,2024,705706501,71,96,30,23,1,0,1,0,...,0,,9,,,5,,,2,-.430807838924267
37388,2024,705707301,,,,,,,,,...,,,,,,,,,,


# 1638 - Peso talla y anemia

In [ ]:
# RECH 6 es la base de niñas y niños en el hogar 
# RECH 5 esla base de mujeres de 12 a 49 años
# RECH 44 Es una base de niñas y niños en el cuestionario individual (no usaremos esta)


In [18]:
ruta_rech6 = base / "1638" / "RECH6_2024.csv"
ruta_rech5 = base / "1638" / "RECH5_2024.csv"

rech6 = pd.read_csv(ruta_rech6, dtype=str)
rech5 = pd.read_csv(ruta_rech5, dtype=str)

In [19]:
rech5

,ID1,HHID,HA0,HA1,HA2,HA3,HA4,HA5,HA6,HA11,...,HA60,HA61,HA62,HA63,HA64,HA65,HA66,HA67,HA68,HA69
0,2024,325503101,2,38,704,1525,301,-188,9315,108,...,,,,,,1,2,5,2,
1,2024,325503101,4,15,541,1518,591,-156,9355,16,...,,,,,,1,2,3,2,
2,2024,325503901,2,17,498,1602,2931,-54,9798,-101,...,,,,,,1,2,5,2,
3,2024,325504701,2,32,653,1443,6,-325,8814,146,...,,,,,,1,2,5,2,
4,2024,325505001,1,36,689,1486,57,-253,9077,122,...,,,,,,1,1,6,1,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38483,2024,705705001,6,30,707,1635,4861,-3,9987,60,...,,,,,,1,2,3,2,
38484,2024,705705001,7,12,494,1575,6430,37,10159,-143,...,,,,,,1,1,6,1,
38485,2024,705706501,2,35,584,1489,65,-248,9095,7,...,,,,,,1,2,5,2,
38486,2024,705708801,1,39,652,1422,2,-361,8686,134,...,,,,,,1,2,3,2,


# 1640

In [23]:
# CSALUD01_2024: Es la base del cuestionario de salud para personas de 15 años a más.
# CSALUD08_2024: Es la base de salud bucal, ocular y mental en niñas y niños de 0 a 11 años.
# USAREMOS AMBOS

In [21]:
ruta_SALUD1 = base / "1640" / "CSALUD01_2024.csv"
ruta_SALUD8 = base / "1640" / "CSALUD08_2024.csv"

SALUD1 = pd.read_csv(ruta_SALUD1, dtype=str)
SALUD8 = pd.read_csv(ruta_SALUD8, dtype=str)

In [22]:
SALUD1

,ID1,HHID,QHCLUSTER,QHNUMBER,QHHOME,QSNUMERO,QSINTM,QSINTY,QSTOTVISIT,QSRESULT,...,QS731_E,QS731_EA,QS731_F,QS731_G,QS731_H,QS731_I,QS731_J,QS731_X,QS731_Y,PESO15_AMAS
0,2024,325503101,3255,31,1,2,5,2024,1,1,...,,,,,,,,,,157182.172077703
1,2024,325503901,3255,39,1,2,5,2024,1,1,...,,,,,,,,,,200845.215117424
2,2024,325504001,3255,40,1,1,5,2024,1,1,...,,,,,,,,,,432165.467953967
3,2024,325504701,3255,47,1,1,5,2024,1,1,...,,,,,,,,,,162338.353517329
4,2024,325505001,3255,50,1,2,5,2024,2,1,...,,,,,,,,,,162338.353517329
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34013,2024,705703901,7057,39,1,1,12,2024,1,1,...,,,,,,,,,,413006.910789338
34014,2024,705704301,7057,43,1,2,12,2024,3,1,...,,,,,,,,,,885502.426143606
34015,2024,705705001,7057,50,1,2,12,2024,5,1,...,,,,,,,,,,379954.605473915
34016,2024,705706501,7057,65,1,2,12,2024,1,1,...,,,,,,,,,,413006.910789338
